# Forecast: Random Forest Flight Price Prediction

This is the main standalone notebook for flight price forecasting.

It keeps only the highest-impact columns and removes duplicated/leaky columns:
- keep `depart_hour`, remove `departure_time`;
- keep `depart_dow`, remove `depart_is_weekend`;
- keep `search_month`, remove `search_season`;
- keep `arrival_time`, because it describes arrival convenience;
- remove `price_per_km`, `price_per_hour`, `revenue_proxy`, `price_segment`, because they are derived from `price`.

Run the cells from top to bottom.


In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [2]:
# Paths
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'etl' / 'exports').exists():
    repo_root = repo_root.parent

INPUT_CSV = repo_root / 'etl' / 'exports' / 'flights_features_all.csv'
OUTPUT_DIR = repo_root / 'backend' / 'ml' / 'models' / 'forecast_random_forest'

# Use None to read the full file. For a fast check, set LIMIT_ROWS = 5000.
LIMIT_ROWS = None

# Use 0 to train on all cleaned rows.
SAMPLE_SIZE = 0

TEST_SIZE = 0.2
RANDOM_STATE = 42
N_ESTIMATORS = 140
MAX_DEPTH = 22
MIN_SAMPLES_LEAF = 10
MAX_FEATURES = 0.55
N_JOBS = 1  # Use -1 locally only if your Windows permissions allow it.

TARGET_COLUMN = 'price'

NUMERIC_FEATURES = [
    'days_to_departure',
    'stops',
    'duration_minutes',
    'distance_km',
    'depart_hour',
]

CATEGORICAL_FEATURES = [
    'travel_class',
    'airline',
    'origin',
    'destination',
    'arrival_time',
    'search_month',
    'depart_dow',
    'depart_month',
]

FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
ID_COLUMNS = ['flight_id', 'departure_date', 'origin', 'destination', 'airline', 'travel_class']
READ_COLUMNS = list(dict.fromkeys(FEATURE_COLUMNS + [TARGET_COLUMN] + ID_COLUMNS))

REMOVED_COLUMNS = [
    'flight_id',
    'departure_date',
    'departure_time',
    'passengers_total',
    'trip_type',
    'origin_type',
    'destination_type',
    'depart_is_weekend',
    'depart_season',
    'search_dow',
    'search_is_weekend',
    'search_season',
    'price_per_km',
    'price_per_hour',
    'revenue_proxy',
    'price_segment',
]

print('Input:', INPUT_CSV)
print('Output:', OUTPUT_DIR)
print('Features:', FEATURE_COLUMNS)


Input: C:\Users\PC\Desktop\Graduation-Work\etl\exports\flights_features_all.csv
Output: C:\Users\PC\Desktop\Graduation-Work\backend\ml\models\forecast_random_forest
Features: ['days_to_departure', 'stops', 'duration_minutes', 'distance_km', 'depart_hour', 'travel_class', 'airline', 'origin', 'destination', 'arrival_time', 'search_month', 'depart_dow', 'depart_month']


In [3]:
def load_dataset(path: Path, limit_rows: int | None = None) -> pd.DataFrame:
    return pd.read_csv(
        path,
        low_memory=False,
        nrows=limit_rows,
        usecols=lambda column: str(column).strip() in READ_COLUMNS,
    )


def normalize_and_clean(frame: pd.DataFrame) -> pd.DataFrame:
    df = frame.copy()
    df.columns = [str(column).strip() for column in df.columns]

    missing = [column for column in FEATURE_COLUMNS + [TARGET_COLUMN] if column not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    for column in NUMERIC_FEATURES + [TARGET_COLUMN]:
        df[column] = pd.to_numeric(df[column], errors='coerce')

    for column in CATEGORICAL_FEATURES:
        df[column] = (
            df[column]
            .astype('string')
            .str.strip()
            .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, 'null': pd.NA})
            .fillna('Unknown')
        )

    valid_mask = (
        df[TARGET_COLUMN].gt(0)
        & df['days_to_departure'].ge(0)
        & df['distance_km'].gt(0)
        & df['duration_minutes'].gt(0)
        & df['stops'].ge(0)
    )
    keep_columns = list(dict.fromkeys(FEATURE_COLUMNS + [TARGET_COLUMN] + [c for c in ID_COLUMNS if c in df.columns]))
    return df.loc[valid_mask, keep_columns].dropna(subset=[TARGET_COLUMN])


raw = load_dataset(INPUT_CSV, LIMIT_ROWS)
df = normalize_and_clean(raw)

if SAMPLE_SIZE and SAMPLE_SIZE > 0 and len(df) > SAMPLE_SIZE:
    df_model = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)
else:
    df_model = df

print('Rows in file/read:', len(raw))
print('Rows after cleaning:', len(df))
print('Rows used for model:', len(df_model))
display(df_model[FEATURE_COLUMNS + [TARGET_COLUMN]].head())


Rows in file/read: 5150048
Rows after cleaning: 5119917
Rows used for model: 5119917


,days_to_departure,stops,duration_minutes,distance_km,depart_hour,travel_class,airline,origin,destination,arrival_time,search_month,depart_dow,depart_month,price
0,0,1.0,760.0,1246.083665,19.0,Business Class,KLM,KRK,CDG,Morning,3,0,3,673.0
1,0,1.0,745.0,1246.083665,21.0,Business Class,LOT,KRK,CDG,Morning,3,0,3,799.0
2,0,2.0,680.0,1246.083665,21.0,Business Class,"LOT, Air Baltic",KRK,CDG,Morning,3,0,3,1065.0
3,0,1.0,795.0,1246.083665,19.0,Business Class,KLM,KRK,CDG,Morning,3,0,3,673.0
4,0,1.0,1275.0,1246.083665,21.0,Business Class,LOT,KRK,CDG,Evening,3,0,3,1009.0


In [ ]:
try:
    encoder = OneHotEncoder(
        handle_unknown='infrequent_if_exist',
        min_frequency=25,
        sparse_output=True,
    )
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', SimpleImputer(strategy='median'), NUMERIC_FEATURES),
        (
            'categorical',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', encoder),
                ]
            ),
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder='drop',
)

model = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        (
            'model',
            RandomForestRegressor(
                n_estimators=N_ESTIMATORS,
                max_depth=MAX_DEPTH,
                min_samples_leaf=MIN_SAMPLES_LEAF,
                max_features=MAX_FEATURES,
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            ),
        ),
    ]
)

X = df_model[FEATURE_COLUMNS]
y = np.log1p(df_model[TARGET_COLUMN].astype(float))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

model.fit(X_train, y_train)
predicted_log = model.predict(X_test)
actual_price = np.expm1(y_test)
predicted_price = np.maximum(np.expm1(predicted_log), 0)

metrics = {
    'mae': float(mean_absolute_error(actual_price, predicted_price)),
    'rmse': float(np.sqrt(mean_squared_error(actual_price, predicted_price))),
    'r2': float(r2_score(actual_price, predicted_price)),
    'mape_pct': float((np.abs(actual_price - predicted_price) / actual_price.replace(0, np.nan)).mean() * 100.0),
    'model': 'RandomForestRegressor',
    'target': 'log1p(price)',
    'rows_in_file_or_read': int(len(raw)),
    'rows_after_cleaning': int(len(df)),
    'rows_used_for_model': int(len(df_model)),
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'refit_on_full_data': True,
    'final_fit_rows': int(len(df_model)),
    'features': FEATURE_COLUMNS,
    'removed_columns': REMOVED_COLUMNS,
}

# Save the exported artifact after refitting on every cleaned row used in this run.
model.fit(X, y)

print(json.dumps(metrics, indent=2, ensure_ascii=False))


In [ ]:
def aggregate_feature_importance(model: Pipeline) -> pd.DataFrame:
    preprocessor = model.named_steps['preprocess']
    forest = model.named_steps['model']
    feature_names = preprocessor.get_feature_names_out()
    detailed = pd.DataFrame({
        'encoded_feature': feature_names,
        'importance': forest.feature_importances_,
    })
    cleaned_names = detailed['encoded_feature'].astype(str).str.replace(
        r'^(numeric|categorical)__', '', regex=True
    )
    rows = []
    for feature in FEATURE_COLUMNS:
        mask = cleaned_names.eq(feature) | cleaned_names.str.startswith(feature + '_')
        rows.append({'feature': feature, 'importance': float(detailed.loc[mask, 'importance'].sum())})
    return pd.DataFrame(rows).sort_values('importance', ascending=False)


def build_prediction_table() -> pd.DataFrame:
    result = df_model.loc[X_test.index, [c for c in ID_COLUMNS if c in df_model.columns]].copy()
    actual = actual_price.to_numpy()
    predicted = predicted_price
    result['actual_price'] = np.round(actual, 2)
    result['predicted_price'] = np.round(predicted, 2)
    result['absolute_error'] = np.round(np.abs(actual - predicted), 2)
    result['absolute_error_pct'] = np.round(
        np.where(actual > 0, np.abs(actual - predicted) / actual * 100.0, np.nan),
        2,
    )
    return result


feature_importance = aggregate_feature_importance(model)
predictions = build_prediction_table()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        'model': model,
        'target_column': TARGET_COLUMN,
        'feature_columns': FEATURE_COLUMNS,
        'numeric_features': NUMERIC_FEATURES,
        'categorical_features': CATEGORICAL_FEATURES,
    },
    OUTPUT_DIR / 'forecast_random_forest_model.joblib',
)
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8')
feature_importance.to_csv(OUTPUT_DIR / 'feature_importance.csv', index=False, encoding='utf-8')
predictions.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False, encoding='utf-8')

print('Saved to:', OUTPUT_DIR)
display(feature_importance)
display(predictions.head(20))


Saved to: C:\Users\PC\Desktop\Graduation-Work\backend\ml\models\forecast_random_forest


,feature,importance
5,travel_class,0.496906
6,airline,0.193580
1,stops,0.145084
0,days_to_departure,0.035343
7,origin,0.034301
2,duration_minutes,0.024143
3,distance_km,0.017924
8,destination,0.017771
4,depart_hour,0.012470
10,search_month,0.009468


,flight_id,departure_date,origin,destination,airline,travel_class,actual_price,predicted_price,absolute_error,absolute_error_pct
2459656,e6a08b8e6efa2ff817035ea849800b976c04c43e72b138...,2026-03-19,LHR,NCE,Iberia,Business Class,777.0,654.62,122.38,15.75
2677202,a34c4cf779e0899ca383a986b35e04e186f3ab27b20668...,2026-03-14,LHR,VCE,British Airways,Economy Class,257.0,180.02,76.98,29.95
1963396,108b9e66b7465629a739b6d59578bac7d13dfef31cd310...,2026-05-08,KRK,NCE,"Air Dolomiti, Lufthansa",Business Class,1024.0,722.17,301.83,29.48
3778343,572bc4490be4e88d08f53f8de53f3cc9d57258975153ca...,2026-04-28,WAW,VCE,"LOT, Lufthansa",Business Class,808.0,1451.60,643.60,79.65
1522279,278a7eac44243c8b1950a8f3293d95582ab6507585ae3a...,2026-04-15,LHR,MAD,"British Airways, Iberia",Economy Class,337.0,382.96,45.96,13.64
1650638,d2b117b5930a7b13b77342e2e636bd2c0b29074de25f8f...,2026-04-29,WAW,ORY,"LOT, Tap Air Portugal",Economy Class,758.0,396.10,361.90,47.74
550112,a03ac7ccf81feca466329580c8d332318d13a122fb2a6b...,2026-03-25,WAW,MRS,"LOT, Lufthansa",Business Class,636.0,671.09,35.09,5.52
3427403,004e04679e4671c6845cf858f519bf1dee6254854acd04...,2026-04-15,LHR,MRS,Turkish Airlines,Economy Class,356.0,498.19,142.19,39.94
2313144,995387a7337418f1672df58dd27bf51d83f5775ad24b80...,2026-03-02,LGW,ORY,Vueling,Economy Class,73.0,62.56,10.44,14.30
3029878,42d46fed284ec7edfb3e17012a19b1a0bb954b50747cb2...,2026-03-31,WAW,MAD,"LOT, Lufthansa City Airlines",Economy Class,526.0,381.18,144.82,27.53
